# 1. Perform naive bayes classification from scratch with 5 datasets from kaggle.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import math

df1 = pd.read_csv('iris_mini.csv')
df2 = pd.read_csv('titanic_mini.csv')
df3 = pd.read_csv('breast_cancer.csv')
df4 = pd.read_csv('framingham.csv')
df5 = pd.read_csv('Housing.csv')

print("df1 head:\n", df1.head())
print("df2 head:\n", df2.head())
print("df3 head:\n", df3.head())
print("df4 head:\n", df4.head())
print("df5 head:\n", df5.head())


class GaussianNaiveBayes:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.n_classes = len(self.classes)
        self.n_features = X.shape[1]

        self._priors = np.zeros(self.n_classes)
        self._means = np.zeros((self.n_classes, self.n_features))
        self._variances = np.zeros((self.n_classes, self.n_features))  # Use variance instead of std dev

        for idx, c in enumerate(self.classes):
            X_c = X[y == c]
            self._priors[idx] = len(X_c) / len(X)
            # Adding a small epsilon to variance to prevent division by zero
            self._variances[idx, :] = X_c.var(axis=0) + 1e-6
            self._means[idx, :] = X_c.mean(axis=0)

    def _calculate_likelihood(self, x, class_idx, feature_idx):
        mean = self._means[class_idx, feature_idx]
        variance = self._variances[class_idx, feature_idx]

        # Gaussian probability density function
        numerator = np.exp(-((x - mean) ** 2) / (2 * variance))
        denominator = np.sqrt(2 * math.pi * variance)
        return numerator / denominator

    def _calculate_log_posterior(self, x, class_idx):
        log_prior = np.log(self._priors[class_idx])
        log_likelihood = 0
        for feature_idx in range(self.n_features):
            likelihood = self._calculate_likelihood(x[feature_idx], class_idx, feature_idx)
            # Add a small epsilon to likelihood to prevent log(0) errors
            log_likelihood += np.log(likelihood + 1e-10)
        return log_prior + log_likelihood

    def predict(self, X):
        predictions = []
        X_np = X.values if isinstance(X, pd.DataFrame) else X
        for x_sample in X_np:
            log_posteriors = [self._calculate_log_posterior(x_sample, idx) for idx in range(self.n_classes)]
            predicted_class_idx = np.argmax(log_posteriors)
            predictions.append(self.classes[predicted_class_idx])
        return np.array(predictions)

# Function to run Naive Bayes for given datasets
def run_naive_bayes(df, name, feature_cols, target_col, handle_nan=False, target_transform=None):
    print(f"\nRunning Naive Bayes for {name}")

    df_cleaned = df.copy()

    if handle_nan:
        initial_rows = len(df_cleaned)
        df_cleaned.dropna(inplace=True)
        if df_cleaned.empty:
            print(f"Skipping {name}: No data left after dropping NaNs.")
            return
        print(f"NaNs handled for {name}: Original rows {initial_rows}, Cleaned rows {len(df_cleaned)}")

    if target_transform:
        df_cleaned[target_col] = target_transform(df_cleaned[target_col])

    X = df_cleaned[feature_cols]
    y = df_cleaned[target_col]

    if len(np.unique(y)) < 2:
        print(f"Skipping {name}: Target variable '{target_col}' has less than 2 unique classes after preprocessing.")
        return

    # Final check for null values in selected features after all preprocessing
    if X.isnull().any().any():
        print(f"Skipping {name}: Still NaNs present in features after preprocessing. Columns with NaNs:\n{X.isnull().sum()[X.isnull().sum() > 0]}")
        return

    # Use stratify if there are multiple classes to ensure class distribution in splits
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    except ValueError as e:
        print(f"Warning for {name}: Could not stratify split. {e}. Using non-stratified split.")
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    model = GaussianNaiveBayes()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy for {name}: {accuracy:.4f}")

#Apply Naive Bayes
run_naive_bayes(df1, "Iris Dataset", ['petal_length', 'petal_width'], 'species')

run_naive_bayes(df2, "Titanic Dataset", ['Sex', 'Children', 'FirstClass'], 'Survived')

def map_breast_cancer_class(series):
    return series.map({2: 0, 4: 1})
breast_cancer_features = [col for col in df3.columns if col != 'Class']
run_naive_bayes(df3, "Breast Cancer Dataset", breast_cancer_features, 'Class', target_transform=map_breast_cancer_class)

framingham_features = [col for col in df4.columns if col != 'TenYearCHD']
run_naive_bayes(df4, "Framingham Dataset", framingham_features, 'TenYearCHD', handle_nan=True)

def binarize_price(series):
    return (series > series.median()).astype(int)
housing_features = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']
run_naive_bayes(df5, "Housing Dataset", housing_features, 'price', target_transform=binarize_price)


df1 head:
    petal_length  petal_width  species
0           1.4          0.2        0
1           1.4          0.2        0
2           1.3          0.2        0
3           1.5          0.2        0
4           1.4          0.2        0
df2 head:
    Sex  Children  FirstClass  Survived
0    0         0           0         0
1    1         0           1         1
2    1         0           0         1
3    1         0           1         1
4    0         0           0         0
df3 head:
    Clump Thickness  Uniformity of Cell Size  Uniformity of Cell Shape  \
0                5                        1                         1   
1                5                        4                         4   
2                3                        1                         1   
3                6                        8                         8   
4                4                        1                         1   

   Marginal Adhesion  Single Epithelial Cell Size  Bare Nuclei  \


# 2. Perform logistic regression from scratch with 2 kaggle datasets


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

df6 = pd.read_csv('breast_cancer.csv')
df7 = pd.read_csv('framingham.csv')

class LogisticRegression:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        # Gradient Descent
        for _ in range(self.n_iterations):
            # Linear model output
            linear_model = np.dot(X, self.weights) + self.bias
            y_predicted = self._sigmoid(linear_model)

            #gradients
            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / n_samples) * np.sum(y_predicted - y)

            # Update weights and bias
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

    def predict(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        y_predicted = self._sigmoid(linear_model)
        y_predicted_cls = [1 if i > 0.5 else 0 for i in y_predicted]
        return np.array(y_predicted_cls)

# Function to run Logistic Regression
def run_logistic_regression(df, name, feature_cols, target_col, handle_nan=False, target_transform=None):
    print(f"\nLogistic Regression for {name}")

    df_cleaned = df.copy()

    if handle_nan:
        initial_rows = len(df_cleaned)
        df_cleaned.dropna(inplace=True)
        if df_cleaned.empty:
            print(f"Skipping {name}: No data left after dropping NaNs.")
            return
        print(f"NaNs handled for {name}: Original rows {initial_rows}, Cleaned rows {len(df_cleaned)}")
    if target_transform:
        df_cleaned[target_col] = target_transform(df_cleaned[target_col])

    X = df_cleaned[feature_cols]
    y = df_cleaned[target_col]

    if len(np.unique(y)) < 2:
        print(f"Skipping {name}: Target variable '{target_col}' has less than 2 unique classes after preprocessing.")
        return
    if X.isnull().any().any():
        print(f"Skipping {name}: Still NaNs present in features after preprocessing. Columns with NaNs:\n{X.isnull().sum()[X.isnull().sum() > 0]}")
        return

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)

    try:
        X_train, X_test, y_train, y_test = train_test_split(X_scaled_df, y, test_size=0.3, random_state=42, stratify=y)
    except ValueError as e:
        print(f"Warning for {name}: Could not stratify split. {e}. Using non-stratified split.")
        X_train, X_test, y_train, y_test = train_test_split(X_scaled_df, y, test_size=0.3, random_state=42)

    model = LogisticRegression(learning_rate=0.01, n_iterations=1000)
    model.fit(X_train.values, y_train.values) # Convert to numpy arrays for custom model
    y_pred = model.predict(X_test.values)

    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy for {name}: {accuracy:.4f}")

#Logistic Regression

def map_breast_cancer_class(series):
    return series.map({2: 0, 4: 1})
breast_cancer_features_lr = [col for col in df6.columns if col != 'Class']
run_logistic_regression(df6, "Breast Cancer Dataset (df6)", breast_cancer_features_lr, 'Class', target_transform=map_breast_cancer_class)

framingham_features_lr = [col for col in df7.columns if col != 'TenYearCHD']
run_logistic_regression(df7, "Framingham Dataset (df7)", framingham_features_lr, 'TenYearCHD', handle_nan=True)



Logistic Regression for Breast Cancer Dataset (df6)
Accuracy for Breast Cancer Dataset (df6): 0.9610

Logistic Regression for Framingham Dataset (df7)
NaNs handled for Framingham Dataset (df7): Original rows 4238, Cleaned rows 3656
Accuracy for Framingham Dataset (df7): 0.8523


# 3. Perform polynomial regression from scratch using housing dataset.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

df8 = pd.read_csv('Housing.csv')

class PolynomialRegression:
    def __init__(self, degree=2, learning_rate=0.01, n_iterations=1000):
        self.degree = degree
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None

    def _create_polynomial_features(self, X):
        # This approach does not include interaction terms for simplicity in 'from scratch' implementation.
        poly_X = X.copy()
        for d in range(2, self.degree + 1):
            # Handle potential overflow for very large powers
            current_poly_term = X**d
            # Replace inf with large finite number or 0, or check for nan before hstack
            current_poly_term[np.isinf(current_poly_term)] = np.finfo(np.float64).max # Cap large values
            if np.any(np.isnan(current_poly_term)):
                print(f"Warning: NaN generated in polynomial features for degree {d}")
                # Can choose to fill NaNs or stop, for now we will just let it propagate to diagnose

            poly_X = np.hstack((poly_X, current_poly_term))
        return poly_X

    def fit(self, X, y):
        #polynomial features and add bias (intercept) term
        X_poly = self._create_polynomial_features(X)

        # Check for NaNs/Infs after feature creation
        if np.any(np.isnan(X_poly)) or np.any(np.isinf(X_poly)):
            print("Error: NaNs or Infs detected in polynomial features before fitting. Model may fail.")
            # As a basic handling, replace NaNs with 0, or inf with a large finite number
            X_poly = np.nan_to_num(X_poly, nan=0.0, posinf=np.finfo(np.float64).max, neginf=np.finfo(np.float64).min)

        n_samples, n_features = X_poly.shape
        X_poly = np.insert(X_poly, 0, 1, axis=1) # Add bias (intercept) term

        # Initialize weights (including bias term)
        self.weights = np.zeros(n_features + 1)

        # Gradient Descent
        for i in range(self.n_iterations):
            y_predicted = np.dot(X_poly, self.weights)
            errors = y_predicted - y

            # Update weights
            gradient = (1/n_samples) * np.dot(X_poly.T, errors)
            # Prevent exploding gradients/weights by checking for NaNs/Infs in gradient
            if np.any(np.isnan(gradient)) or np.any(np.isinf(gradient)):
                print(f"Warning: NaN or Inf in gradient at iteration {i}. Stopping training.")
                break
            self.weights -= self.learning_rate * gradient

    def predict(self, X):
        X_poly = self._create_polynomial_features(X)
        # Ensure consistent handling of NaNs/Infs in prediction as well
        if np.any(np.isnan(X_poly)) or np.any(np.isinf(X_poly)):
            X_poly = np.nan_to_num(X_poly, nan=0.0, posinf=np.finfo(np.float64).max, neginf=np.finfo(np.float64).min)

        X_poly = np.insert(X_poly, 0, 1, axis=1) # Add bias (intercept) term
        return np.dot(X_poly, self.weights)

#Data Preprocessing for df8
df = df8.copy()

#binary to numerical
binary_map = {'yes': 1, 'no': 0}
df['mainroad'] = df['mainroad'].map(binary_map)
df['guestroom'] = df['guestroom'].map(binary_map)
df['basement'] = df['basement'].map(binary_map)
df['hotwaterheating'] = df['hotwaterheating'].map(binary_map)
df['airconditioning'] = df['airconditioning'].map(binary_map)
df['prefarea'] = df['prefarea'].map(binary_map)

df = pd.get_dummies(df, columns=['furnishingstatus'], drop_first=True)

# Define features (X) and target (y)
X = df.drop('price', axis=1)
y = df['price']

# Check for and handle any remaining Null Values in features, though there shouldn't be any after preprocessing
if X.isnull().any().any():
    print("Warning: NaNs found in features after preprocessing. Dropping rows with NaNs.")
    nan_rows = X.isnull().any(axis=1)
    X = X[~nan_rows]
    y = y[~nan_rows]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale numerical features (important for gradient descent)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#Polynomial Regression
degrees = [1, 2, 3]
results = {}

for degree in degrees:
    print(f"\nPolynomial Regression (Degree = {degree})")
    # Adjust learning_rate and n_iterations.
    if degree == 3:
        model = PolynomialRegression(degree=degree, learning_rate=0.001, n_iterations=5000) # Reduced learning rate
    else:
        model = PolynomialRegression(degree=degree, learning_rate=0.01, n_iterations=2000)

    model.fit(X_train_scaled, y_train.values)
    y_pred = model.predict(X_test_scaled)

    # Check if y_pred contains Null values
    if np.any(np.isnan(y_pred)):
        print(f"Skipping metrics for degree {degree} due to NaN in predictions.")
        results[degree] = {'MSE': np.nan, 'RMSE': np.nan, 'R2': np.nan}
        continue

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    results[degree] = {'MSE': mse, 'RMSE': rmse, 'R2': r2}
    print(f"Mean Squared Error: {mse:.2f}")
    print(f"Root Mean Squared Error: {rmse:.2f}")
    print(f"R-squared: {r2:.4f}")


Polynomial Regression (Degree = 1)
Mean Squared Error: 1522999812746.20
Root Mean Squared Error: 1234098.79
R-squared: 0.6463

Polynomial Regression (Degree = 2)
Mean Squared Error: 1579630179245.66
Root Mean Squared Error: 1256833.39
R-squared: 0.6332

Polynomial Regression (Degree = 3)
Mean Squared Error: 1743579482540.46
Root Mean Squared Error: 1320446.70
R-squared: 0.5951
